# Bingo Tutorial 4: Symbolic Regression

This tutorial evolves an expression by targeting input-output data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-10, 10, 30).reshape(-1, 1)
y = x**2 + 3.5 * x**3
plt.plot(x, y, 'ro')
plt.show()

## Expression generator and variation

`AGraphGenerator` produces evolvable expressions. The minimum and maximum stack sizes bound the size of generated and crossed-over expressions.  Note that even if the minimum stack size is >>1, you can still have small expressions result (meaning much of the stack is unutilized).

In [ ]:
from bingo.expressions import (
    AGraphCrossover, AGraphGenerator, AGraphMutation, ComponentGenerator,
)

component_generator = ComponentGenerator(x.shape[1], random_state=3)
for operator in ('+', '-', '*'):
    component_generator.add_operator(operator)

MIN_STACK_SIZE = 10
MAX_STACK_SIZE = 10
generator = AGraphGenerator(MIN_STACK_SIZE, MAX_STACK_SIZE, component_generator, random_state=3)
crossover = AGraphCrossover(MIN_STACK_SIZE, MAX_STACK_SIZE, random_state=3)
mutation = AGraphMutation(component_generator, random_state=3)
individuals = [generator() for _ in range(3)]
for individual in individuals:
    print(individual)

## Explicit regression

`ExplicitRegression` fits an expression's constants on first evaluation, then reports its loss. Predictions can be obtained directly from the resulting expression.

In [ ]:
from bingo.symbolic_regression import ExplicitRegression

objective = ExplicitRegression(x, y)
plt.plot(x, y, 'ro')
for individual in individuals:
    print(individual, ' fitness:', objective(individual))
    prediction = individual.predict(x)
    plt.plot(x, prediction, '-')
plt.show()

## Evolution and a Pareto front

An island evaluates and evolves the expressions. The Pareto front retains the fitness-complexity tradeoff.

In [ ]:
from bingo.evaluation.evaluation import Evaluation
from bingo.evolutionary_algorithms.age_fitness import AgeFitnessEA
from bingo.evolutionary_optimizers.island import Island
from bingo.stats.pareto_front import ParetoFront

population_size = 32
evaluator = Evaluation(objective)
ea = AgeFitnessEA(evaluator, generator, crossover, mutation, 0.4, 0.4, population_size)
pareto_front = ParetoFront(
    secondary_key=lambda individual: individual.complexity,
    similarity_function=lambda first, second: first == second,
)
island = Island(ea, generator, population_size, hall_of_fame=pareto_front)
island.evolve_until_convergence(max_generations=500, fitness_threshold=1e-6)

print('Pareto front:')
print('fitness          complexity       expression')
for member in pareto_front:
    print(f'{member.fitness:.3e}  {member.complexity:^20d}  {member}')

In [ ]:
plt.plot(x, y, 'ro')
for member in pareto_front:
    prediction = member.predict(x)
    plt.plot(x, prediction, label=f'fitness: {member.fitness:.3e}, complexity: {member.complexity}')
plt.legend()
plt.show()